In [1]:
# Cell 1 — Imports
import os
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS

# Optional but very useful for HEALPix FITS sky maps
try:
    import healpy as hp
    HAS_HEALPY = True
except Exception:
    HAS_HEALPY = False

plt.rcParams["figure.figsize"] = (10, 6)


In [2]:
# Cell 2 — Set your FITS path
fits_path = "../magic_2024c_fig2_skymap.fits"  # <-- change this

if not os.path.exists(fits_path):
    raise FileNotFoundError(f"File not found: {fits_path}")


In [3]:
# Cell 3 — Quick FITS inspection (what HDUs exist? what shapes?)
with fits.open(fits_path) as hdul:
    hdul.info()
    for i, hdu in enumerate(hdul):
        if hdu.data is None:
            continue
        print(f"\nHDU {i}: {type(hdu).__name__}")
        print("  data type:", type(hdu.data))
        # hdu.data can be numpy array or FITS_rec (table)
        if hasattr(hdu.data, "shape"):
            print("  shape:", hdu.data.shape)
        if hasattr(hdu, "columns") and hdu.columns is not None:
            print("  columns:", hdu.columns.names)


Filename: ../magic_2024c_fig2_skymap.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      12   ()      
  1  THETA2PLOT    1 BinTableHDU     25   69R x 4C   [1D, 1D, 1D, 1D]   

HDU 1: BinTableHDU
  data type: <class 'astropy.io.fits.fitsrec.FITS_rec'>
  shape: (69,)
  columns: ['theta2', 'Non', 'Noff', 'DNoff']


In [4]:
# Cell 4 — Helper: try to detect whether this looks like a HEALPix sky map table
def looks_like_healpix_table(hdu):
    # Common for HEALPix: a BinTableHDU with a column containing the map, e.g. 'PROB', 'SIGNAL', etc.
    if not hasattr(hdu, "columns") or hdu.columns is None:
        return False
    colnames = [c.upper() for c in hdu.columns.names]
    # Heuristic: presence of a probability-like column OR typical HEALPix keywords in header
    header = hdu.header
    has_hp_kw = any(k in header for k in ["NSIDE", "ORDERING", "INDXSCHM"])
    has_prob_col = any(n in colnames for n in ["PROB", "PROBABILITY", "SIGNAL", "MAP", "VALUE", "PIXEL", "UNIQ"])
    return has_hp_kw or has_prob_col


In [5]:
# Cell 5 — Display function for WCS 2D image HDU
def show_wcs_image(hdu, title=None, cmap="viridis", vmin=None, vmax=None):
    data = hdu.data
    if data is None:
        raise ValueError("No data in this HDU.")
    # If data has extra axes, take the first plane
    if data.ndim > 2:
        data2d = np.squeeze(data)
        if data2d.ndim != 2:
            # fall back: pick first 2D plane
            data2d = data.reshape((-1, *data.shape[-2:]))[0]
    else:
        data2d = data

    w = WCS(hdu.header)

    fig = plt.figure()
    ax = plt.subplot(projection=w)
    im = ax.imshow(data2d, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid(color="white", ls=":", alpha=0.4)
    plt.colorbar(im, ax=ax, label="Value")
    ax.set_title(title or "Sky map (WCS image)")
    plt.show()


In [6]:
# Cell 6 — Display function for HEALPix map (requires healpy)
def show_healpix_map_from_hdu(hdu, title=None, nest=None, field=0):
    if not HAS_HEALPY:
        raise ImportError("healpy is not installed. Install with: pip install healpy")

    data = hdu.data
    header = hdu.header

    # Determine ordering: NESTED vs RING
    ordering = header.get("ORDERING", "").strip().upper()
    if nest is None:
        if ordering == "NESTED":
            nest = True
        elif ordering == "RING":
            nest = False
        else:
            # default assumption if not present
            nest = False

    # Common cases:
    # - table has a column with the map, often first column or named PROB, MAP, etc.
    colnames = [c for c in hdu.columns.names]
    colnames_upper = [c.upper() for c in colnames]

    preferred = ["PROB", "PROBABILITY", "MAP", "SIGNAL", "VALUE"]
    chosen_col = None
    for p in preferred:
        if p in colnames_upper:
            chosen_col = colnames[colnames_upper.index(p)]
            break
    if chosen_col is None:
        # fallback to first column
        chosen_col = colnames[0]

    m = data[chosen_col]
    # If column is multi-dimensional, pick a field
    if hasattr(m, "ndim") and m.ndim > 1:
        m = m[:, field]

    # Ensure it's a plain numpy array
    m = np.array(m)

    # Plot all-sky view (mollweide)
    hp.mollview(m, nest=nest, title=title or f"HEALPix sky map ({chosen_col})", unit="Value")
    hp.graticule()
    plt.show()


In [7]:
# Cell 7 — Main: pick an HDU and display intelligently
with fits.open(fits_path) as hdul:
    # Pick the first HDU that has data
    candidate_indices = [i for i, h in enumerate(hdul) if h.data is not None]
    if not candidate_indices:
        raise ValueError("No HDU with data found in this FITS file.")

    # Try to find a good display HDU:
    # Prefer an IMAGE HDU for WCS; otherwise look for HEALPix-like table
    chosen = None

    # 1) First image-like
    for i in candidate_indices:
        if isinstance(hdul[i], (fits.ImageHDU, fits.PrimaryHDU)) and hasattr(hdul[i].data, "ndim"):
            if hdul[i].data.ndim >= 2:
                chosen = i
                break

    # 2) Otherwise HEALPix-like table
    if chosen is None:
        for i in candidate_indices:
            if isinstance(hdul[i], fits.BinTableHDU) and looks_like_healpix_table(hdul[i]):
                chosen = i
                break

    # 3) Fallback to first with data
    if chosen is None:
        chosen = candidate_indices[0]

    hdu = hdul[chosen]
    print(f"Displaying HDU {chosen}: {type(hdu).__name__}")

    if isinstance(hdu, (fits.ImageHDU, fits.PrimaryHDU)) and hasattr(hdu.data, "ndim") and hdu.data.ndim >= 2:
        show_wcs_image(hdu, title=f"{os.path.basename(fits_path)} — HDU {chosen}")
    elif isinstance(hdu, fits.BinTableHDU) and looks_like_healpix_table(hdu):
        show_healpix_map_from_hdu(hdu, title=f"{os.path.basename(fits_path)} — HEALPix (HDU {chosen})")
    else:
        raise ValueError(
            "Don't know how to display this HDU automatically. "
            "It is neither a 2D image nor a recognizable HEALPix table."
        )


Displaying HDU 1: BinTableHDU


ValueError: Don't know how to display this HDU automatically. It is neither a 2D image nor a recognizable HEALPix table.